# PHASE 5: Model Selection, Training & Validation
**Traceability**
- Issue ID: #5 Model Selection, Training & Validation

## 1. Objectives
- Establish a leakage-safe validation strategy using engine-level splitting.
- Implement robust scaling (MinMaxScaler) fitted exclusively on training data.
- Train regression baselines (Linear Regression) and advanced tree models (XGBoost, LightGBM).
- Evaluate performance using RMSE, MAE, R², and the NASA asymmetric scoring function.
- Train binary classifiers for early failure warning.

### 5.1 Import Libraries & Configure Training
We import modeling libraries and define the NASA scoring function to penalize late predictions more heavily.

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, classification_report, confusion_matrix
import xgboost as xgb
import lightgbm as lgb
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# ── Reproducibility Config ──────────────────────────────────────────────
np.random.seed(42)

# ── Global Config ────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../data/processed')
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

def nasa_score(y_true, y_pred):
    """NASA asymmetric scoring function."""
    d = y_pred - y_true
    scores = np.where(d >= 0, np.exp(d / 13) - 1, np.exp(-d / 10) - 1)
    return np.sum(scores)

def regression_report(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    ns   = nasa_score(y_true, y_pred)
    print(f"  {name:<35} RMSE={rmse:.2f}  MAE={mae:.2f}  R²={r2:.3f}  NASA={ns:.1f}")
    return {'model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'NASA_Score': ns}

### 5.2 Data Splitting & Preprocessing
Split engines into training and validation sets to prevent data leakage. Scale features using only the training distribution.

In [ ]:
# 1. Load Labeled Data
df_train = pd.read_csv(PROCESSED_DIR / 'train_labeled.csv')
df_test = pd.read_csv(PROCESSED_DIR / 'test_labeled.csv')

# 2. Engine-Level Split (80/20)
all_units = df_train['unit_number'].unique()
np.random.shuffle(all_units)
n_train = int(0.80 * len(all_units))
train_units = all_units[:n_train]
val_units = all_units[n_train:]

df_tr = df_train[df_train['unit_number'].isin(train_units)].copy()
df_va = df_train[df_train['unit_number'].isin(val_units)].copy()

# 3. Scaling
feature_cols = [c for c in df_train.columns if not any(x in c for x in ['unit_number', 'RUL', 'label'])]
scaler = MinMaxScaler()
X_tr = scaler.fit_transform(df_tr[feature_cols])
X_va = scaler.transform(df_va[feature_cols])
X_te = scaler.transform(df_test[feature_cols])

# For "Last Cycle" evaluation on test set
df_test_last = df_test.groupby('unit_number').last().reset_index()
X_te_last = scaler.transform(df_test_last[feature_cols])
y_te_last_rul = df_test_last['RUL'].values

y_tr_rul = df_tr['RUL'].values
y_va_rul = df_va['RUL'].values
y_te_rul = df_test['RUL'].values

with open(ARTIFACTS_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

### 5.3 RUL Regression Modeling
Train and compare several regression models. We evaluate them on both the full test set and the last known cycles of each engine.

In [ ]:
print("\n--- Regression Task (Test Set Metrics) ---")

# Linear Regression
lr = LinearRegression()
lr.fit(X_tr, y_tr_rul)
regression_report("Linear Regression (All Cycles)", y_te_rul, lr.predict(X_te))
regression_report("Linear Regression (Last Cycle)", y_te_last_rul, lr.predict(X_te_last))

# XGBoost
xgb_reg = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1)
xgb_reg.fit(X_tr, y_tr_rul, eval_set=[(X_va, y_va_rul)], verbose=False)
regression_report("XGBoost (All Cycles)", y_te_rul, xgb_reg.predict(X_te))
regression_report("XGBoost (Last Cycle)", y_te_last_rul, xgb_reg.predict(X_te_last))

# LightGBM
lgb_reg = lgb.LGBMRegressor(n_estimators=200, max_depth=5, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
lgb_reg.fit(X_tr, y_tr_rul, eval_set=[(X_va, y_va_rul)], callbacks=[lgb.early_stopping(20, verbose=False)])
regression_report("LightGBM (All Cycles)", y_te_rul, lgb_reg.predict(X_te))
regression_report("LightGBM (Last Cycle)", y_te_last_rul, lgb_reg.predict(X_te_last))

# Random Forest
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_reg.fit(X_tr, y_tr_rul)
regression_report("Random Forest (All Cycles)", y_te_rul, rf_reg.predict(X_te))
regression_report("Random Forest (Last Cycle)", y_te_last_rul, rf_reg.predict(X_te_last))

# Save best regression model
with open(ARTIFACTS_DIR / 'best_regressor.pkl', 'wb') as f:
    pickle.dump(xgb_reg, f)

### 5.4 Risk Classification Modeling
Train a classifier to predict whether an engine is within its final 30 cycles of life.

In [ ]:
print("\n--- Classification Task (Test Set Metrics) ---")
knn = KNeighborsClassifier(n_neighbors=50, n_jobs=-1)
knn.fit(X_tr, df_tr['label1'].values)
knn_preds = knn.predict(X_te)
print("  KNN (n=50) Classification Report (All Test Cycles):")
print(classification_report(df_test['label1'].values, knn_preds, target_names=['Healthy', 'At Risk']))

# Confusion Matrix Visualization
cm = confusion_matrix(df_test['label1'].values, knn_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Healthy', 'At Risk'], yticklabels=['Healthy', 'At Risk'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: KNN Classifier')
plt.show()

with open(ARTIFACTS_DIR / 'best_classifier.pkl', 'wb') as f:
    pickle.dump(knn, f)